# What is an LLM? Tokens, Embeddings, and the Big Picture

**Companion notebook for the [MLWhiz blog post](https://www.mlwhiz.com/p/what-is-an-llm-tokens-embeddings)**

This notebook walks through every step of the LLM pipeline with runnable code:
1. Tokenization (BPE, SentencePiece, tiktoken)
2. Embedding table lookup
3. Weight tying
4. Temperature sampling
5. Autoregressive generation loop

In [1]:
!pip install tiktoken sentencepiece transformers torch -q

## 1. Tokenization: How Text Becomes Numbers

### tiktoken (GPT-4, Llama 3, Qwen)

In [2]:
import tiktoken

# GPT-4's tokenizer
enc = tiktoken.get_encoding("cl100k_base")

# Simple English
text = "The quick brown fox jumps over the lazy dog"
tokens = enc.encode(text)
print(f"Text: {text}")
print(f"Token IDs: {tokens}")
print(f"Token count: {len(tokens)}")
print(f"Decoded: {[enc.decode([t]) for t in tokens]}")

Text: The quick brown fox jumps over the lazy dog
Token IDs: [791, 4062, 14198, 39935, 35308, 927, 279, 16053, 5679]
Token count: 9
Decoded: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog']


In [3]:
# The strawberry problem!
for word in ["strawberry", "lollipop", "February", "ChatGPT", "unfamiliarize"]:
    toks = enc.encode(word)
    decoded = [enc.decode([t]) for t in toks]
    print(f"'{word}' -> {decoded} ({len(toks)} tokens)")

'strawberry' -> ['str', 'aw', 'berry'] (3 tokens)
'lollipop' -> ['l', 'ollipop'] (2 tokens)
'February' -> ['February'] (1 tokens)
'ChatGPT' -> ['Chat', 'G', 'PT'] (3 tokens)
'unfamiliarize' -> ['un', 'f', 'amiliar', 'ize'] (4 tokens)


In [4]:
# Code tokenization
code = "def fibonacci(n):\n    return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)"
code_tokens = enc.encode(code)
print(f"Code: {code}")
print(f"Token count: {len(code_tokens)}")
print(f"Decoded: {[enc.decode([t]) for t in code_tokens]}")

Code: def fibonacci(n):
    return n if n < 2 else fibonacci(n-1) + fibonacci(n-2)
Token count: 24
Decoded: ['def', ' fibonacci', '(n', '):\n', '   ', ' return', ' n', ' if', ' n', ' <', ' ', '2', ' else', ' fibonacci', '(n', '-', '1', ')', ' +', ' fibonacci', '(n', '-', '2', ')']


In [5]:
# Compare tokenizer vocabularies
encodings = {
    "GPT-2 (gpt2)": tiktoken.get_encoding("gpt2"),
    "GPT-4 (cl100k_base)": tiktoken.get_encoding("cl100k_base"),
    "GPT-4o (o200k_base)": tiktoken.get_encoding("o200k_base"),
}

test_texts = [
    "Hello, how are you?",
    "The quick brown fox",
    "def fibonacci(n): return n",
    "strawberry",
]

print(f"{'Encoding':<25} {'Vocab Size':>12}")
print("-" * 40)
for name, e in encodings.items():
    print(f"{name:<25} {e.n_vocab:>12,}")

print()
for text in test_texts:
    print(f"\nText: '{text}'")
    for name, e in encodings.items():
        toks = e.encode(text)
        print(f"  {name:<25} {len(toks)} tokens: {[e.decode([t]) for t in toks]}")

Encoding                    Vocab Size
----------------------------------------
GPT-2 (gpt2)                    50,257
GPT-4 (cl100k_base)            100,277
GPT-4o (o200k_base)            200,019


Text: 'Hello, how are you?'
  GPT-2 (gpt2)              6 tokens: ['Hello', ',', ' how', ' are', ' you', '?']
  GPT-4 (cl100k_base)       6 tokens: ['Hello', ',', ' how', ' are', ' you', '?']
  GPT-4o (o200k_base)       6 tokens: ['Hello', ',', ' how', ' are', ' you', '?']

Text: 'The quick brown fox'
  GPT-2 (gpt2)              4 tokens: ['The', ' quick', ' brown', ' fox']
  GPT-4 (cl100k_base)       4 tokens: ['The', ' quick', ' brown', ' fox']
  GPT-4o (o200k_base)       4 tokens: ['The', ' quick', ' brown', ' fox']

Text: 'def fibonacci(n): return n'
  GPT-2 (gpt2)              9 tokens: ['def', ' fib', 'on', 'acci', '(', 'n', '):', ' return', ' n']
  GPT-4 (cl100k_base)       6 tokens: ['def', ' fibonacci', '(n', '):', ' return', ' n']
  GPT-4o (o200k_base)       6 tokens: ['def', ' fi

### Comparing tokenizers across model families

Different models use different tokenizer implementations. The space character gets encoded differently:
- **tiktoken / GPT-style BPE**: spaces become part of the token (` how` with a leading space)
- **SentencePiece**: spaces are replaced with `▁` (e.g., `▁how`)
- **HuggingFace BPE** (Qwen, GPT-2): spaces appear as `Ġ` internally (the byte-level representation of a leading space)

All three mean the same thing — "this token starts after a space." The notation is just different.

In [6]:
from transformers import AutoTokenizer

# Compare multiple tokenizers side by side
tokenizers = {
    "Qwen 3 0.6B": AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B"),
    "Qwen 2.5 0.5B": AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B"),
    "Mistral 7B": AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1"),
}

text = "Hello, how are you?"
print(f"Text: '{text}'\n")

for name, tok in tokenizers.items():
    tokens = tok.tokenize(text)
    ids = tok.encode(text)
    print(f"{name} (vocab={tok.vocab_size:,}):")
    print(f"  Tokens: {tokens}")
    print(f"  IDs:    {ids}")
    if any('Ġ' in t for t in tokens):
        print(f"  Note: Ġ = leading space (byte-level BPE)")
    elif any('▁' in t for t in tokens):
        print(f"  Note: ▁ = leading space (SentencePiece)")
    print()

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Text: 'Hello, how are you?'

Qwen 3 0.6B (vocab=151,643):
  Tokens: ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']
  IDs:    [9707, 11, 1246, 525, 498, 30]
  Note: Ġ = leading space (byte-level BPE)

Qwen 2.5 0.5B (vocab=151,643):
  Tokens: ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']
  IDs:    [9707, 11, 1246, 525, 498, 30]
  Note: Ġ = leading space (byte-level BPE)

Mistral 7B (vocab=32,000):
  Tokens: ['▁Hello', ',', '▁how', '▁are', '▁you', '?']
  IDs:    [1, 22557, 28725, 910, 460, 368, 28804]
  Note: ▁ = leading space (SentencePiece)



## 2. The Embedding Table: How Numbers Become Vectors

In [7]:
import torch
import torch.nn as nn

# Create an embedding table (same as a real model, just smaller)
vocab_size = 128_256
d_model = 4096
embedding = nn.Embedding(vocab_size, d_model)

# Look up a single token
token_id = torch.tensor([7234])
vector = embedding(token_id)
print(f"Token ID: {token_id.item()}")
print(f"Vector shape: {vector.shape}")
print(f"First 10 values: {vector[0, :10].tolist()}")

# Look up a sequence
input_ids = torch.tensor([791, 4062, 14198, 39935])  # "The quick brown fox"
vectors = embedding(input_ids)
print(f"\nSequence of 4 tokens -> shape: {vectors.shape}")

# Parameter count
params = vocab_size * d_model
print(f"\nEmbedding table parameters: {params:,} ({params/1e6:.1f}M)")

Token ID: 7234
Vector shape: torch.Size([1, 4096])
First 10 values: [-0.5829929113388062, 1.8207809925079346, 1.2299246788024902, 0.12553976476192474, 0.7695755958557129, -0.24098902940750122, -0.9413871169090271, 0.04441229999065399, -0.8705067038536072, 1.2649171352386475]

Sequence of 4 tokens -> shape: torch.Size([4, 4096])

Embedding table parameters: 525,336,576 (525.3M)


In [8]:
# Embedding table sizes across models
models = [
    ("GPT-2", 50_257, 768, 124_000_000),
    ("Mistral 7B", 32_000, 4096, 7_200_000_000),
    ("Llama 3 8B", 128_256, 4096, 8_000_000_000),
    ("Qwen 2.5 7B", 151_643, 3584, 7_600_000_000),
    ("Gemma 2 9B", 256_000, 3584, 9_200_000_000),
]

print(f"{'Model':<18} {'Vocab':>10} {'d_model':>8} {'Embed Params':>14} {'% of Total':>10}")
print("-" * 65)
for name, v, d, total in models:
    embed_params = v * d
    pct = embed_params / total * 100
    print(f"{name:<18} {v:>10,} {d:>8} {embed_params:>14,} {pct:>9.1f}%")

Model                   Vocab  d_model   Embed Params % of Total
-----------------------------------------------------------------
GPT-2                  50,257      768     38,597,376      31.1%
Mistral 7B             32,000     4096    131,072,000       1.8%
Llama 3 8B            128,256     4096    525,336,576       6.6%
Qwen 2.5 7B           151,643     3584    543,488,512       7.2%
Gemma 2 9B            256,000     3584    917,504,000      10.0%


## 3. Weight Tying: Same Matrix, Two Jobs

In [9]:
# Tiny worked example: vocab=5, d_model=3
# Tokens: "the"=0, "cat"=1, "sat"=2, "on"=3, "mat"=4

E = torch.tensor([
    [0.9, 0.1, 0.5],  # "the"
    [0.2, 0.8, 0.1],  # "cat"
    [0.1, 0.6, 0.9],  # "sat"
    [0.7, 0.3, 0.4],  # "on"
    [0.5, 0.4, 0.7],  # "mat"
])
token_names = ["the", "cat", "sat", "on", "mat"]

print("=== Job 1: Embedding ===")
print(f"Token 'cat' (ID=1) -> look up row 1 of E")
cat_embedding = E[1]
print(f"Embedding: {cat_embedding.tolist()}")

print("\n=== Job 2: LM Head (weight tying) ===")
h = torch.tensor([0.15, 0.65, 0.80])  # hidden state from transformer
print(f"Hidden state h = {h.tolist()}")
print(f"\nDot product of h with each row of E:")
logits = E @ h  # matrix-vector multiply = dot product with each row
for i, (name, logit) in enumerate(zip(token_names, logits)):
    marker = " <-- highest" if logit == logits.max() else ""
    print(f"  '{name}': {' + '.join(f'{h[j]:.2f}x{E[i,j]:.1f}' for j in range(3))} = {logit:.2f}{marker}")

print(f"\nLogits: {logits.tolist()}")
probs = torch.softmax(logits, dim=0)
print(f"Softmax: {[f'{p:.2f}' for p in probs.tolist()]}")
print(f"Predicted: '{token_names[probs.argmax()]}'")

=== Job 1: Embedding ===
Token 'cat' (ID=1) -> look up row 1 of E
Embedding: [0.20000000298023224, 0.800000011920929, 0.10000000149011612]

=== Job 2: LM Head (weight tying) ===
Hidden state h = [0.15000000596046448, 0.6499999761581421, 0.800000011920929]

Dot product of h with each row of E:
  'the': 0.15x0.9 + 0.65x0.1 + 0.80x0.5 = 0.60
  'cat': 0.15x0.2 + 0.65x0.8 + 0.80x0.1 = 0.63
  'sat': 0.15x0.1 + 0.65x0.6 + 0.80x0.9 = 1.12 <-- highest
  'on': 0.15x0.7 + 0.65x0.3 + 0.80x0.4 = 0.62
  'mat': 0.15x0.5 + 0.65x0.4 + 0.80x0.7 = 0.89

Logits: [0.6000000238418579, 0.6299999356269836, 1.125, 0.6200000047683716, 0.8949999809265137]
Softmax: ['0.16', '0.17', '0.28', '0.17', '0.22']
Predicted: 'sat'


In [10]:
# Without vs with weight tying: parameter savings
for name, v, d, total in models:
    embed_params = v * d
    no_tie = embed_params * 2  # embedding + separate LM head
    with_tie = embed_params    # shared
    saved = embed_params
    print(f"{name:<18} Without tying: {no_tie/1e6:>8.1f}M | With tying: {with_tie/1e6:>8.1f}M | Saved: {saved/1e6:>8.1f}M")

GPT-2              Without tying:     77.2M | With tying:     38.6M | Saved:     38.6M
Mistral 7B         Without tying:    262.1M | With tying:    131.1M | Saved:    131.1M
Llama 3 8B         Without tying:   1050.7M | With tying:    525.3M | Saved:    525.3M
Qwen 2.5 7B        Without tying:   1087.0M | With tying:    543.5M | Saved:    543.5M
Gemma 2 9B         Without tying:   1835.0M | With tying:    917.5M | Saved:    917.5M


## 4. Temperature Sampling

In [11]:
import torch.nn.functional as F

# Same logits from our tiny example
logits = torch.tensor([0.60, 0.63, 1.12, 0.62, 0.89])
token_names = ["the", "cat", "sat", "on", "mat"]

print(f"{'Temperature':<15} {'the':>8} {'cat':>8} {'sat':>8} {'on':>8} {'mat':>8}  Winner")
print("-" * 80)

for T in [0.1, 0.5, 1.0, 2.0, 5.0]:
    probs = F.softmax(logits / T, dim=0)
    winner = token_names[probs.argmax()]
    prob_strs = [f"{p:.3f}" for p in probs.tolist()]
    print(f"T = {T:<10} {prob_strs[0]:>8} {prob_strs[1]:>8} {prob_strs[2]:>8} {prob_strs[3]:>8} {prob_strs[4]:>8}  {winner}")

print("\nT -> 0: always picks 'sat' (greedy, deterministic)")
print("T = 1: natural distribution")
print("T -> inf: uniform random (all tokens equally likely)")

Temperature          the      cat      sat       on      mat  Winner
--------------------------------------------------------------------------------
T = 0.1           0.005    0.007    0.893    0.006    0.090  sat
T = 0.5           0.130    0.138    0.367    0.135    0.231  sat
T = 1.0           0.165    0.170    0.277    0.168    0.220  sat
T = 2.0           0.183    0.185    0.237    0.184    0.211  sat
T = 5.0           0.193    0.194    0.214    0.194    0.205  sat

T -> 0: always picks 'sat' (greedy, deterministic)
T = 1: natural distribution
T -> inf: uniform random (all tokens equally likely)


## 5. Autoregressive Generation Loop

Using a real model from HuggingFace to see the full pipeline in action.

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load Qwen 3 0.6B — small enough to run on free Kaggle GPU
model_name = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)

print(f"Model: {model_name}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

# Check embedding table size
embed_params = model.model.embed_tokens.weight.numel()
total_params = sum(p.numel() for p in model.parameters())
print(f"Embedding params: {embed_params:,} ({embed_params/total_params*100:.1f}% of total)")
print(f"Embedding shape: {model.model.embed_tokens.weight.shape}")
print(f"d_model: {model.config.hidden_size}")
print(f"n_layers: {model.config.num_hidden_layers}")
print(f"n_heads: {model.config.num_attention_heads}")

# Also load Qwen 2.5 0.5B for the weight tying comparison later
model_name_v2 = "Qwen/Qwen2.5-0.5B"
tokenizer_v2 = AutoTokenizer.from_pretrained(model_name_v2)
model_v2 = AutoModelForCausalLM.from_pretrained(model_name_v2, torch_dtype=torch.float16)
print(f"\nAlso loaded: {model_name_v2} (for weight tying comparison)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model: Qwen/Qwen3-0.6B
Vocab size: 151,643
Total params: 751,632,384
Embedding params: 155,582,464 (20.7% of total)
Embedding shape: torch.Size([151936, 1024])
d_model: 1024
n_layers: 28
n_heads: 16


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Also loaded: Qwen/Qwen2.5-0.5B (for weight tying comparison)


In [13]:
# Step-by-step generation: see every token being produced
prompt = "The cat sat on"
input_ids = tokenizer.encode(prompt, return_tensors="pt")
print(f"Prompt: '{prompt}'")
print(f"Input IDs: {input_ids[0].tolist()}")
print(f"Tokens: {[tokenizer.decode([t]) for t in input_ids[0]]}")
print()

# Generate token by token
generated = input_ids[0].tolist()
temperature = 0.8

for step in range(10):
    with torch.no_grad():
        outputs = model(torch.tensor([generated]))
        logits = outputs.logits[0, -1, :]  # last position
    
    # Apply temperature and get top-5
    probs = torch.softmax(logits / temperature, dim=0)
    top5 = torch.topk(probs, 5)
    
    # Sample
    next_token = torch.multinomial(probs, 1).item()
    next_word = tokenizer.decode([next_token])
    
    print(f"Step {step+1}: top-5 = {[(tokenizer.decode([t.item()]), f'{p:.3f}') for t, p in zip(top5.indices, top5.values)]}")
    print(f"         sampled: '{next_word}' (token {next_token})")
    
    generated.append(next_token)
    
    if next_token == tokenizer.eos_token_id:
        print("         [EOS - stopping]")
        break

print(f"\nFull output: '{tokenizer.decode(generated)}'")

Prompt: 'The cat sat on'
Input IDs: [785, 8251, 7578, 389]
Tokens: ['The', ' cat', ' sat', ' on']

Step 1: top-5 = [(' the', '0.806'), (' a', '0.174'), (' top', '0.007'), (' my', '0.007'), (' his', '0.002')]
         sampled: ' the' (token 279)
Step 2: top-5 = [(' edge', '0.132'), (' window', '0.098'), (' floor', '0.097'), (' chair', '0.076'), (' bed', '0.072')]
         sampled: ' edge' (token 6821)
Step 3: top-5 = [(' of', '0.996'), (',', '0.002'), ('.', '0.001'), (' and', '0.000'), ('.\n\n', '0.000')]
         sampled: ' of' (token 315)
Step 4: top-5 = [(' the', '0.851'), (' a', '0.146'), (' her', '0.001'), (' his', '0.001'), (' an', '0.001')]
         sampled: ' the' (token 279)
Step 5: top-5 = [(' bed', '0.173'), (' table', '0.146'), (' bench', '0.131'), (' couch', '0.047'), (' chair', '0.039')]
         sampled: ' table' (token 1965)
Step 6: top-5 = [(',', '0.512'), ('.', '0.258'), (' and', '0.154'), (' in', '0.019'), ('.\n\n', '0.012')]
         sampled: ',' (token 11)
Step 7: t

In [14]:
# Weight tying: compare Qwen 2.5 (has it) vs Qwen 3 (doesn't)

print("=" * 60)
print("Qwen 2.5 0.5B")
print("=" * 60)
embed_w = model_v2.model.embed_tokens.weight
lm_w = model_v2.lm_head.weight
print(f"Embedding shape: {embed_w.shape}")
print(f"LM head shape:   {lm_w.shape}")
tied = embed_w.data_ptr() == lm_w.data_ptr()
print(f"Same object in memory: {tied}")
if tied:
    print("✅ Weight tying! Embedding and LM head share the same matrix.")
    print(f"   Saves {embed_w.numel():,} parameters ({embed_w.numel()/1e6:.0f}M)")
else:
    print("❌ No weight tying. Separate embedding and LM head matrices.")
    print(f"   Extra cost: {lm_w.numel():,} parameters ({lm_w.numel()/1e6:.0f}M)")

print()
print("=" * 60)
print("Qwen 3 0.6B")
print("=" * 60)
embed_w3 = model.model.embed_tokens.weight
lm_w3 = model.lm_head.weight
print(f"Embedding shape: {embed_w3.shape}")
print(f"LM head shape:   {lm_w3.shape}")
tied3 = embed_w3.data_ptr() == lm_w3.data_ptr()
print(f"Same object in memory: {tied3}")
if tied3:
    print("✅ Weight tying! Embedding and LM head share the same matrix.")
    print(f"   Saves {embed_w3.numel():,} parameters ({embed_w3.numel()/1e6:.0f}M)")
else:
    print("❌ No weight tying. Separate embedding and LM head matrices.")
    print(f"   Extra cost: {lm_w3.numel():,} parameters ({lm_w3.numel()/1e6:.0f}M)")

print()
print("Not all models tie weights. Qwen 3 chose separate matrices,")
print("possibly because the embedding and output distributions benefit")
print("from learning different representations at this model size.")

Qwen 2.5 0.5B
Embedding shape: torch.Size([151936, 896])
LM head shape:   torch.Size([151936, 896])
Same object in memory: True
✅ Weight tying! Embedding and LM head share the same matrix.
   Saves 136,134,656 parameters (136M)

Qwen 3 0.6B
Embedding shape: torch.Size([151936, 1024])
LM head shape:   torch.Size([151936, 1024])
Same object in memory: False
❌ No weight tying. Separate embedding and LM head matrices.
   Extra cost: 155,582,464 parameters (156M)

Not all models tie weights. Qwen 3 chose separate matrices,
possibly because the embedding and output distributions benefit
from learning different representations at this model size.


## 6. Multilingual Tokenization: Why Vocab Size Matters

In [15]:
# Compare token counts for the same meaning in different languages
enc = tiktoken.get_encoding("cl100k_base")  # GPT-4

texts = {
    "English": "Hello, how are you doing today?",
    "Hindi": "नमस्ते, आज आप कैसे हैं?",
    "Japanese": "こんにちは、今日はお元気ですか？",
    "Chinese": "你好，你今天过得怎么样？",
    "Korean": "안녕하세요, 오늘 어떻게 지내세요?",
    "Arabic": "مرحبا، كيف حالك اليوم؟",
}

print(f"{'Language':<12} {'Chars':>6} {'Tokens':>7} {'Ratio':>7}")
print("-" * 35)
en_tokens = len(enc.encode(texts["English"]))
for lang, text in texts.items():
    n_chars = len(text)
    n_tokens = len(enc.encode(text))
    ratio = n_tokens / en_tokens
    print(f"{lang:<12} {n_chars:>6} {n_tokens:>7} {ratio:>6.1f}x")

print(f"\nSame meaning, wildly different token counts.")
print(f"More tokens = higher cost, shorter effective context, slower inference.")

Language      Chars  Tokens   Ratio
-----------------------------------
English          31       8    1.0x
Hindi            23      24    3.0x
Japanese         16      11    1.4x
Chinese          12      13    1.6x
Korean           19      19    2.4x
Arabic           22      18    2.2x

Same meaning, wildly different token counts.
More tokens = higher cost, shorter effective context, slower inference.
